In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "font.size": 14,
    "axes.labelsize": 14,
    "axes.titlesize": 14,
    "legend.fontsize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
})

In [ ]:
%cd ~/GraphFEX
%cd ~/blue/chunmei.wang/hudsonshields/GraphFEX

controller_epochs = 5
cands_per_epoch = 10

small_num_nodes = [20, 50, 500]
order_of_mag_nodes = [10, 100, 1000]
node_counts = sorted(set(small_num_nodes + order_of_mag_nodes))

timing_results = {}

pct_edges = 0.1
for n in node_counts:
    degree = max(1, int(pct_edges * n))

    # K = 0.5d = 0.05n, giving approximately 20 nodes per batch.
    desired_batches = max(1, int(0.1 * degree))

    t1 = %timeit -o -r5 -n1 !python -m NumericalExperiments.HR.scripts.eval_controller_dim_x \
        --num_workers 1 \
        --num_epochs 60 \
        --controller_epochs {controller_epochs} \
        --nodes {n} \
        --num_cands_per_epoch {cands_per_epoch} \
        --num_stoch_batches {desired_batches} \
        --pct_edges {pct_edges}

    timing_results[n] = {
        "mean": t1.average / controller_epochs / cands_per_epoch,
        "std": t1.stdev / controller_epochs / cands_per_epoch,
    }

In [ ]:

control_timing_results = {}

for n in node_counts:
    t1 = %timeit -o -r5 -n1 !python -m NumericalExperiments.HR.scripts.eval_controller_dim_x \
        --num_workers 1 \
        --num_epochs 60 \
        --controller_epochs {controller_epochs} \
        --nodes {n} \
        --num_cands_per_epoch {cands_per_epoch} \
        --pct_edges {pct_edges}

    control_timing_results[n] = {
        "mean": t1.average / controller_epochs / cands_per_epoch,
        "std": t1.stdev / controller_epochs / cands_per_epoch,
    }


In [ ]:
parallel_timing_results = {}
parallel_workers = 2

for n in node_counts:
    t1 = %timeit -o -r5 -n1 !python -m NumericalExperiments.HR.scripts.eval_controller_dim_x \
        --num_workers {parallel_workers} \
        --num_epochs 60 \
        --controller_epochs {controller_epochs} \
        --nodes {n} \
        --num_cands_per_epoch {cands_per_epoch} \
        --pct_edges {pct_edges}

    parallel_timing_results[n] = {
        "mean": t1.average / controller_epochs / cands_per_epoch,
        "std": t1.stdev / controller_epochs / cands_per_epoch,
    }

In [ ]:
par_stoch_timing_results = {}

for n in node_counts:
    degree = max(1, int(pct_edges * n))

    # K = 0.5d = 0.05n, giving approximately 20 nodes per batch.
    desired_batches = max(1, int(0.1 * degree))

    t1 = %timeit -o -r5 -n1 !python -m NumericalExperiments.HR.scripts.eval_controller_dim_x \
        --num_workers {parallel_workers} \
        --num_epochs 60 \
        --controller_epochs {controller_epochs} \
        --nodes {n} \
        --num_cands_per_epoch {cands_per_epoch} \
        --num_stoch_batches {desired_batches} \
        --pct_edges {pct_edges}

    par_stoch_timing_results[n] = {
        "mean": t1.average / controller_epochs / cands_per_epoch,
        "std": t1.stdev / controller_epochs / cands_per_epoch,
    }

%cd ~/GraphFEX
%cd /blue/chunmei.wang/hudsonshields/GraphFEX

In [ ]:
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",

    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

experiment_labels = [
    "Serial",
    "Stochastic",
    "Parallel",
    "Parallel+Stochastic"
]

experiments = [
    control_timing_results,
    timing_results,
    parallel_timing_results,
    par_stoch_timing_results
]

colors = {
    "Serial": "black",
    "Stochastic": "orange",
    "Parallel": "blue",
    "Parallel+Stochastic": "green",
}

styles = {
    "Serial": {
        "marker": "o",
        "linestyle": "-"
    },
    "Stochastic": {
        "marker": "s",
        "linestyle": "-"
    },
    "Parallel": {
        "marker": "^",
        "linestyle": "--"
    },
    "Parallel+Stochastic": {
        "marker": "D",
        "linestyle": "--"
    },
}

fig, (ax_main, ax_speedup) = plt.subplots(
    1,
    2,
    figsize=(9, 4.5),
    sharex=True
)

# (a) Runtime scaling

for exp, label in zip(experiments, experiment_labels):

    means = np.array([
        exp[n]["mean"]
        for n in node_counts
    ])

    stds = np.array([
        exp[n]["std"]
        for n in node_counts
    ])

    ax_main.plot(
        node_counts,
        means,
        label=label,
        color=colors[label],
        marker=styles[label]["marker"],
        linestyle=styles[label]["linestyle"],
        linewidth=2,
        markersize=5,
    )

    ax_main.fill_between(
        node_counts,
        means - stds,
        means + stds,
        color=colors[label],
        alpha=0.15,
    )

ax_main.set_xscale("log")

ax_main.set_xticks(order_of_mag_nodes)
ax_main.set_xticklabels(
    [
        rf"$10^{{{int(np.log10(n))}}}$"
        for n in order_of_mag_nodes
    ]
)

ax_main.set_xlabel(r"Network Size $n$")
ax_main.set_ylabel("Mean Candidate Evaluation Time (s)")

ax_main.legend(fontsize=9, frameon=False)
ax_main.grid(alpha=0.2, linestyle="--")

ax_main.text(
    -0.23,
    0.98,
    r"(a)",
    transform=ax_main.transAxes,
    # fontsize=12,
    # fontweight="bold",
    va="top"
)


# (b) Speedup relative to serial
serial_means = np.array([control_timing_results[n]["mean"] for n in node_counts])
serial_stds = np.array([control_timing_results[n]["std"] for n in node_counts])


speedup_experiments = [
    ("Stochastic", timing_results),
    ("Parallel", parallel_timing_results),
    ("Parallel+Stochastic", par_stoch_timing_results),
]

for label, exp in speedup_experiments:
    method_means = np.array([exp[n]["mean"] for n in node_counts])
    method_stds = np.array([exp[n]["std"] for n in node_counts])

    # Speedup relative to serial evaluation
    speedup = serial_means / method_means
    speedup_std = speedup * np.sqrt((serial_stds / serial_means) ** 2 + (method_stds / method_means) ** 2)

    ax_speedup.plot(
        node_counts,
        speedup,
        label=label,
        color=colors[label],
        marker=styles[label]["marker"],
        linestyle=styles[label]["linestyle"],
        linewidth=2,
        markersize=5,
    )

    ax_speedup.fill_between(
        node_counts,
        speedup - speedup_std,
        speedup + speedup_std,
        color=colors[label],
        alpha=0.15,
    )


ax_speedup.axhline(
    1,
    color="black",
    linewidth=1,
    linestyle=":",
    alpha=0.7
)

ax_speedup.set_xscale("log")

ax_speedup.set_xticks(order_of_mag_nodes)
ax_speedup.set_xticklabels(
    [
        rf"$10^{{{int(np.log10(n))}}}$"
        for n in order_of_mag_nodes
    ]
)

ax_speedup.set_xlabel(r"Network Size $n$")
ax_speedup.set_ylabel(r"Speedup over Serial ($\times$)")

ax_speedup.legend(
    fontsize=9,
    frameon=False
)

ax_speedup.grid(
    alpha=0.2,
    linestyle="--"
)

ax_speedup.text(
    -0.2,
    0.98,
    r"(b)",
    transform=ax_speedup.transAxes,
    va="top"
)

plt.tight_layout()
plt.show()

In [ ]:
oops

In [ ]:
import os

# 1. Move Jupyter's working directory into the script folder
%cd ~/GraphFEX/HR/scripts
%cd ~/blue/chunmei.wang/hudsonshields/GraphFEX/HR/scripts


mean_times = []
std_dev_times = []

controller_epochs = 5
for num_workers in range(1, 11):
    # 2. Run the script directly without the folder prefix
    t1 = %timeit -o -r5 -n1 !python HR/scripts/eval_controller_dim_x.py --num_workers {num_workers} --num_epochs 80 --controller_epochs {controller_epochs}

    # scale by number of controller epochs because each timing run includes multiple controller epochs, and i want to find the per epoch time
    mean_times.append(t1.average / controller_epochs)
    std_dev_times.append(t1.stdev / controller_epochs)

# 3. Optional: Move back to your project root when finished
%cd ~/GraphFEX
%cd ~/blue/chunmei.wang/hudsonshields/GraphFEX
plt.errorbar(range(1, 11), mean_times, yerr=std_dev_times)
plt.xlabel(r"Number of Parallel Processes")
plt.ylabel(r"Clock Time (s)")
plt.title(r"Multiprocessor Scaling for 10 Cands (64bs 80ep)")
plt.show()